# Module 38 — RFM Segmentation with Pandas

> **Track 4 · Marketing Analytics & Optimization**  
> **Difficulty**: Intermediate  
> **Runtime**: ~5 minutes  
> **Key Libraries**: Pandas, Scikit-learn, Plotly  
> **Prerequisite**: Module 2 (Advanced GroupBy)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic E-commerce Data](#3-synthetic-e-commerce-data)
4. [RFM Calculation](#4-rfm-calculation)
5. [Customer Segmentation](#5-customer-segmentation)
6. [Visualization](#6-visualization)
7. [Key Business Takeaway](#7-key-business-takeaway)

---
## §1 · Business Context

### Why RFM segmentation?

RFM analysis segments customers by **purchase behavior**:
- **Recency**: How recently they purchased.
- **Frequency**: How often they purchase.
- **Monetary**: How much they spend.

**Applications**:
1. **Targeted marketing**: Personalize campaigns by segment.
2. **Customer retention**: Identify at-risk customers.
3. **Resource allocation**: Focus on high-value segments.

**Key segments**:
- **Champions**: Recent, frequent, high spenders.
- **Loyal**: Frequent buyers with good spend.
- **At Risk**: Previously active, now declining.
- **Lost**: Haven't purchased in long time.

In this notebook we will:
1. Generate synthetic e-commerce transaction data.
2. Calculate RFM scores for each customer.
3. Segment customers using KMeans clustering.
4. Visualize segments in 3D scatter plot.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── Scikit-learn ─────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# ── Visualization ────────────────────────────────────────────────
import plotly.graph_objects as go
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("✓ All imports loaded")

---
## §3 · Synthetic E-commerce Data

In [ ]:
# Generate synthetic e-commerce data
N_CUSTOMERS = 10000
N_TRANSACTIONS = 100000

# Generate transactions
transactions = pd.DataFrame({
    'customer_id': np.random.randint(0, N_CUSTOMERS, N_TRANSACTIONS),
    'transaction_date': pd.date_range('2023-01-01', periods=N_TRANSACTIONS, freq='1h'),
    'amount': np.random.exponential(50, N_TRANSACTIONS),
    'product_category': np.random.choice(['electronics', 'clothing', 'books', 'home'], N_TRANSACTIONS)
})

# Calculate reference date
reference_date = transactions['transaction_date'].max()

print(f"✓ Generated {N_TRANSACTIONS:,} transactions")
print(f"  Customers: {N_CUSTOMERS:,}")
print(f"  Date range: {transactions['transaction_date'].min()} to {reference_date}")
print(f"  Total revenue: ${transactions['amount'].sum():,.2f}")

---
## §4 · RFM Calculation

In [ ]:
# Calculate RFM metrics
rfm = transactions.groupby('customer_id').agg({
    'transaction_date': lambda x: (reference_date - x.max()).days,  # Recency
    'customer_id': 'count',  # Frequency
    'amount': 'sum'  # Monetary
}).rename(columns={
    'transaction_date': 'recency',
    'customer_id': 'frequency',
    'amount': 'monetary'
})

print(f"✓ RFM metrics calculated")
print(f"\nRFM Summary:")
print(rfm.describe())
print(f"\nSample customers:")
print(rfm.head())

---
## §5 · Customer Segmentation

In [ ]:
# Scale RFM features
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm)

# Apply KMeans clustering
N_CLUSTERS = 4
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
rfm['segment'] = kmeans.fit_predict(rfm_scaled)

# Name segments based on characteristics
segment_names = {
    0: 'Champions',
    1: 'Loyal',
    2: 'At Risk',
    3: 'Lost'
}

rfm['segment_name'] = rfm['segment'].map(segment_names)

# Segment summary
segment_summary = rfm.groupby('segment_name').agg({
    'recency': 'mean',
    'frequency': 'mean',
    'monetary': ['mean', 'count']
}).round(2)

print(f"✓ Segmentation complete")
print(f"\nSegment Summary:")
print(segment_summary)

---
## §6 · Visualization

In [ ]:
# 3D scatter plot of RFM segments
fig = px.scatter_3d(
    rfm,
    x='recency',
    y='frequency',
    z='monetary',
    color='segment_name',
    title='Customer Segments (RFM)',
    labels={
        'recency': 'Recency (days)',
        'frequency': 'Frequency',
        'monetary': 'Monetary ($)'
    },
    opacity=0.7
)

fig.update_layout(height=600)
fig.show()

print("💡 Key insights:")
print("   - Champions: Recent, frequent, high spenders")
print("   - Loyal: Consistent buyers with good spend")
print("   - At Risk: Previously active, now declining")
print("   - Lost: Haven't purchased recently")

---
## §7 · Key Business Takeaway

> ### 🎯 Action Items for Marketing Teams
>
> | Aspect | Recommendation |
> |--------|---------------|
> | **When to use** | Customer segmentation, targeted campaigns, retention |
> | **Metrics** | Recency, Frequency, Monetary value |
> | **Segments** | 4-6 segments for actionable insights |
> | **Evaluation** | Segment stability, campaign ROI |
> | **Deployment** | Monthly segmentation, real-time scoring |
>
> ### 💡 Marketing Strategies by Segment
>
> | Segment | Strategy | Budget Allocation |
> |---------|----------|------------------|
> | Champions | Reward loyalty, upsell | High |
> | Loyal | Retention programs | Medium |
> | At Risk | Win-back campaigns | Medium |
> | Lost | Deep discounts or let go | Low |
>
> ### 🔑 Key Takeaways
>
> 1. **RFM segmentation is simple yet powerful** for customer analysis.
> 2. **KMeans clustering** provides data-driven segmentation.
> 3. **Different segments require different strategies**.
> 4. **Regular re-segmentation** captures changing behavior.
> 5. **Combine with CLV** for resource allocation.